In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

In [2]:
# Hand detector
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)

# Pose detector
base_options_pose = python.BaseOptions(model_asset_path='pose_landmarker.task')

pose_options = vision.PoseLandmarkerOptions(
    base_options=base_options_pose
)

pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

In [3]:
base_angle = 90
prev_elbow_x = 0

In [4]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # ===== HAND DETECTION =====
    result = detector.detect(mp_image)
    lm_list = []

    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            h, w, _ = frame.shape

            for id, lm in enumerate(hand_landmarks):
                cx, cy = int(lm.x * w), int(lm.y * h)
                lm_list.append((id, cx, cy))
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

    # ===== FINGER DETECTION =====
    fingers = []
    gesture = "None"

    if len(lm_list) >= 21:
        if lm_list[4][1] > lm_list[3][1]:
            fingers.append(1)
        else:
            fingers.append(0)

        tip_ids = [8, 12, 16, 20]
        for tip in tip_ids:
            if lm_list[tip][2] < lm_list[tip - 2][2]:
                fingers.append(1)
            else:
                fingers.append(0)

        if len(fingers) == 5:
            if fingers == [0, 0, 0, 0, 0]:
                gesture = "STOP"
            elif fingers == [0, 1, 0, 0, 0]:
                gesture = "MOVE"
            elif fingers == [0, 1, 1, 0, 0]:
                gesture = "ROTATE"
            elif fingers == [1, 1, 1, 1, 1]:
                gesture = "OPEN"

    # ===== POSE (ELBOW CONTROL) =====
    pose_result = pose_detector.detect(mp_image)

    if pose_result.pose_landmarks:
        h, w, _ = frame.shape
        right_elbow = pose_result.pose_landmarks[0][14]

        elbow_x = int(right_elbow.x * w)
        elbow_y = int(right_elbow.y * h)

        cv2.circle(frame, (elbow_x, elbow_y), 8, (0, 0, 255), -1)

        if prev_elbow_x != 0:
            dx = elbow_x - prev_elbow_x

            # Dead zone
            if abs(dx) < 5:
                dx = 0

            # Scale movement
            rotation = dx * 0.3
            rotation = max(min(rotation, 20), -20)

            base_angle += rotation
            base_angle = max(min(base_angle, 180), 0)

        prev_elbow_x = elbow_x

    else:
        prev_elbow_x = 0

    # ===== COMMAND =====
    command = "NONE"

    if gesture == "MOVE":
        command = f"ROTATE:{int(base_angle)}"
    elif gesture == "STOP":
        command = "STOP"

    # ===== DISPLAY =====
    cv2.putText(frame, f'Gesture: {gesture}', (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f'Base Angle: {int(base_angle)}', (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 100, 0), 2)

    cv2.putText(frame, f'Command: {command}', (10, 170),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    # ===== JUPYTER DISPLAY =====
    cv2.imshow("Hand Tracking", frame)
    # Required for window to update
    if cv2.waitKey(1) & 0xFF == 27:
        break
cap.release()
cv2.destroyAllWindows()